# 00 · Colab setup check (Qwen2.5-7B + NLA)

Run this once per new runtime. It checks that:
1. a GPU is attached (L4 or A100: **Runtime → Change runtime type**),
2. the repo is cloned and installed,
3. your Hugging Face token works (Colab **Secrets** 🔑 → `HF_TOKEN`, with notebook access on),
4. the tokenizer splits `" he"` / `" she"` / `"A"` / `"B"` into **single tokens**. The bias metric depends on this.

No 7B model is loaded here, only its tokenizer.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, sys
REPO = "mi-genderbias-qwen-nla"
if not os.path.exists(f"/content/{REPO}"):
    !git clone -q https://github.com/olawoye/{REPO}.git /content/{REPO}
else:
    !git -C /content/{REPO} pull -q
%cd /content/{REPO}
!pip install -q -e .
sys.path.insert(0, f"/content/{REPO}/src")  # makes the editable install visible without a restart

In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from mibias.config import get_config
from mibias.checkpoint import open_run
cfg = get_config()
run = open_run(cfg)          # checkpoints on Drive; bump run="v2" to start fresh
print("model:", cfg.model_id)
print("checkpoints:", run.root)
print("results:    ", cfg.results_dir)
print("already saved:", run.status() or "nothing yet")

**If Colab disconnects:** reconnect, then *Runtime → Run all*. Finished steps and finished
chunks load from Drive in seconds; only the interrupted chunk is recomputed.
The model weights are re-downloaded each session (~3 min). That's faster than reading 14 GB from Drive.

## Tokenizer check

In [ ]:
from transformers import AutoTokenizer
from mibias import prompts

tok = AutoTokenizer.from_pretrained(cfg.model_id)

# Tokenise *in context*: SentencePiece tokenizers (Mistral) treat a leading space
# differently in isolation, so compare "prefix" vs "prefix + target".
def continuation_ids(prefix, target):
    a = tok.encode(prefix, add_special_tokens=False)
    b = tok.encode(prefix + target, add_special_tokens=False)
    assert b[:len(a)] == a, "prefix tokens changed"
    return b[len(a):]

for prefix, target in [("The nurse said that", prompts.MALE_TOKEN),
                       ("The nurse said that", prompts.FEMALE_TOKEN),
                       ("Answer:", " A"), ("Answer:", " B"), ("Answer:\n", "A"), ("Answer:\n", "B")]:
    ids = continuation_ids(prefix, target)
    print(f"{target!r:6} ->", ids, [tok.decode([i]) for i in ids])
    assert len(ids) == 1, f"{target!r} is not a single token after {prefix!r}"
print("OK: all targets are single tokens in context")

## Prompt sets

In [ ]:
pp, rp = prompts.pronoun_prompts(), prompts.resume_prompts()
print(len(pp), "pronoun prompts | e.g.", repr(pp[0].text))
print(len(rp), "resume prompts  | e.g.\n" + rp[0].text)
print(len(prompts.contrast_pairs()), "contrast pairs for the gender direction")